# JS03: Tugas Lab

Wisconsin Breast Cancer (Again)

## Persiapan data

In [1]:
from pathlib import Path
from urllib.request import Request, urlopen

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
data_path = data_dir / 'wbc.csv'
if not data_path.exists():
    request = Request(
        'https://1812311909-files.gitbook.io/~/files/v0/b/gitbook-x-prod.appspot.com/o/spaces%2FYHOv2XH8FJMJSMsnhwNa%2Fuploads%2Fg1GPNC6t7Dwxntp1SRJk%2Fwbc.csv?alt=media&token=e607f608-87a2-402f-829c-a733241ae84c',
        headers={'User-Agent': 'Mozilla/5.0'}
    )
    data_path.write_bytes(urlopen(request).read())

## Import library

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

## Load data

In [3]:
df = pd.read_csv('data/wbc.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


## Pemisahan variabel

In [4]:
df = df.drop(columns=['id', 'Unnamed: 32'])

X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X.head()

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## Encoding diagnosis

In [5]:
le = LabelEncoder()
y = le.fit_transform(y)

print(le.classes_)
print(np.unique(y))

['B' 'M']
[0 1]


## Pembagian data

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Seleksi jumlah fitur

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for k in range(1, X.shape[1] + 1):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_classif, k=k)),
        ('model', LogisticRegression(max_iter=1000))
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
    results.append((k, scores.mean()))

hasil_seleksi = pd.DataFrame(results, columns=['Jumlah Fitur', 'Akurasi CV'])
hasil_seleksi

,Jumlah Fitur,Akurasi CV
0,1,0.903297
1,2,0.940659
2,3,0.945055
3,4,0.940659
4,5,0.942857
5,6,0.947253
6,7,0.949451
7,8,0.949451
8,9,0.947253
9,10,0.953846


## Jumlah fitur terbaik

In [8]:
best_k = int(hasil_seleksi.loc[hasil_seleksi['Akurasi CV'].idxmax(), 'Jumlah Fitur'])
best_score = hasil_seleksi['Akurasi CV'].max()

print('Jumlah fitur terbaik:', best_k)
print('Akurasi cross validation:', best_score)

Jumlah fitur terbaik: 29
Akurasi cross validation: 0.9736263736263737


## Pipeline final

In [9]:
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=best_k)),
    ('model', LogisticRegression(max_iter=1000))
])

final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)

## Evaluasi model

In [10]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           B       0.96      0.99      0.97        72
           M       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



## Fitur terpilih

In [11]:
selector = final_pipeline.named_steps['selector']
selected_features = X.columns[selector.get_support()]
selected_scores = selector.scores_[selector.get_support()]

fitur_terpilih = pd.DataFrame({
    'Fitur': selected_features,
    'Skor ANOVA': selected_scores
}).sort_values('Skor ANOVA', ascending=False)

print('Jumlah fitur terbaik:', best_k)
print('Fitur yang digunakan:')
display(fitur_terpilih)

Jumlah fitur terbaik: 29
Fitur yang digunakan:


,Fitur,Skor ANOVA
26,concave points_worst,733.724933
21,perimeter_worst,717.246487
19,radius_worst,692.861395
7,concave points_mean,684.526845
2,perimeter_mean,548.413236
22,area_worst,522.188947
0,radius_mean,511.274848
3,area_mean,444.857518
6,concavity_mean,397.592082
25,concavity_worst,319.507776


## Kesimpulan

Hasil 5 fold cross validation memilih 29 fitur dengan rata rata akurasi 97,36%. Model Logistic Regression menghasilkan akurasi 96,49% pada data uji. Daftar fitur yang digunakan ditampilkan pada tabel hasil seleksi fitur. Fitur fractal_dimension_mean tidak terpilih.